# Building Agents with AutoGen

This notebook introduces **Microsoft AutoGen**, a framework for building multi-agent conversations powered by Large Language Models (LLMs). You will learn how to configure an LLM, create conversable agents, and run simple question-answer and code-execution tasks.

---

## Learning objectives

By the end of this notebook you will be able to:

- **Configure an LLM** via a config list (e.g., from a JSON file or environment variables).
- **Create an AssistantAgent** that uses the LLM to generate responses.
- **Create a UserProxyAgent** that represents the user and can optionally execute code.
- **Initiate a chat** between the user proxy and the assistant to complete a task.

---

## Basic workflow (four steps)

1. **Define configuration list** — Specify which model(s) and API keys the agent will use.
2. **Define the assistant** — An LLM-backed agent that reasons and responds (and can suggest code).
3. **Define the user proxy** — Represents the human user; can run code and relay results back.
4. **Initiate chat** — Start a conversation with a task message and let the agents collaborate.

In [7]:
# ---------------------------------------------------------------------------
# Optional: Install AutoGen in your environment (run once per environment)
# ---------------------------------------------------------------------------
# Uncomment the line below if autogen is not already installed.
# pip install autogen-agentchat==0.2.38

In [14]:
# ---------------------------------------------------------------------------
# Imports: Core AutoGen components and utilities
# ---------------------------------------------------------------------------
import autogen
from autogen import AssistantAgent, UserProxyAgent
from autogen.coding import LocalCommandLineCodeExecutor
from pathlib import Path
from IPython.display import Image, Markdown

# AssistantAgent: LLM-powered agent that generates replies (and can suggest code).
# UserProxyAgent: Represents the user; can execute code and return results.
# LLMConfig: Modern API for loading LLM configuration from JSON files.
# LocalCommandLineCodeExecutor: Runs code in a local folder (used later for code execution).

# Verify the installed AutoGen version (helps match tutorial and docs).
autogen.__version__

'0.10.4'

### Step 1: LLM configuration

AutoGen needs to know **which LLM to use** and **how to authenticate**. This is done via a **config list**: a list of model configurations (model name, API key, and optional settings).

**Recommended approach:** Use a `CONFIG_LIST.json` file (or an environment variable holding the same JSON) so that API keys stay out of your notebook and out of version control.

- **Reference:** [LLM configuration (AutoGen docs)](https://microsoft.github.io/autogen/0.2/docs/topics/llm_configuration/)

**Example `CONFIG_LIST.json` format:**

```json
[
  {
    "model": "gpt-4o-mini",
    "api_key": "<your-key-here>"
  }
]
```

For Groq, you would use something like `"model": "llama-3.3-70b-versatile"` and `"api_type": "groq"`, with the key from your environment.

**Security:** Keep API keys out of git (e.g., add `CONFIG_LIST.json` to `.gitignore`) and rotate them regularly.

In [15]:
# ---------------------------------------------------------------------------
# Load environment variables from a .env file (optional but recommended)
# ---------------------------------------------------------------------------
# This allows you to store API keys in .env (e.g. OPENAI_API_KEY, GROQ_API_KEY)
# and reference them in CONFIG_LIST.json via env vars, keeping secrets out of code.
from dotenv import load_dotenv
import os

load_dotenv()  # Loads .env from the current directory (or parent directories)

True

In [16]:
# ---------------------------------------------------------------------------
# Steps 2–4: Load config, create assistant and user proxy, then initiate chat
# ---------------------------------------------------------------------------

# Step 2a: Load model configuration from CONFIG_LIST.json using the modern LLMConfig API.
# Keep this file out of source control; use .env or env vars for the actual API key.
llm_config_obj = autogen.LLMConfig.from_json(path="CONFIG_LIST.json")
config_list = llm_config_obj.config_list

# Optional: Filter config_list by model name
# Example: Filter to use only "llama-3.3-70b-versatile"
# config_list = [config for config in config_list if config.get("model") == "llama-3.3-70b-versatile"]

# Or filter to use only "openai/gpt-oss-120b"
config_list = [config for config in config_list if config.get("model") == "openai/gpt-oss-120b"]

# Or filter by partial model name match
# config_list = [config for config in config_list if "llama" in config.get("model", "").lower()]

In [17]:
# Step 2b: Create the AssistantAgent — the LLM-backed agent that will answer questions.
# - name: Used in conversation logs.
# - system_message: Instructions that shape the assistant’s behavior.
# - llm_config: Tells the agent which model(s) to use (from config_list).
assistant = AssistantAgent(
    name="groq_assistant",
    system_message="You are a helpful AI assistant.",
    llm_config={"config_list": config_list},
    human_input_mode="NEVER"
)

# Step 3: Create the UserProxyAgent — represents the human user in the conversation.
# - code_execution_config=False: This example is Q&A only; no code will be run.
# - human_input_mode="NEVER": no prompts; proxy auto-replies so the chat runs unattended.
# - max_consecutive_auto_reply=1: stop after one assistant reply (avoids endless back-and-forth).
user_proxy = UserProxyAgent(
    name="user_proxy",
    code_execution_config=False,
    human_input_mode="NEVER",
    max_consecutive_auto_reply=1,
)

# Step 4: Start the conversation. The user_proxy sends the first message to the assistant.
# The assistant replies; in this setup you may be prompted for input (e.g. type 'exit' to stop).
user_proxy.initiate_chat(
    assistant,
    message="What are the key benefits of using Groq for AI apps?")

user_proxy (to groq_assistant):

What are the key benefits of using Groq for AI apps?

--------------------------------------------------------------------------------
groq_assistant (to user_proxy):

## Why Groq is becoming a go‑to platform for AI applications

| Benefit | What it means for you | Why it matters |
|---------|----------------------|----------------|
| **Extreme inference speed** | • Single‑cycle tensor streaming → latency measured in micro‑seconds <br>• Deterministic, “no‑surprises” execution time | Real‑time use‑cases (search, recommendation, autonomous control, voice assistants) can meet sub‑ms SLA targets that GPUs or TPUs struggle to guarantee. |
| **Predictable, linear scaling** | • Performance grows linearly with the number of Groq cards <br>• No “GPU‑style” throttling or memory‑bandwidth contention | Capacity planning is simple—add another card and you get a proportional boost, which is rare on conventional accelerators. |
| **Very low power / high efficiency** |

ChatResult(chat_id=328867056625437107388533849703859700183, chat_history=[{'content': 'What are the key benefits of using Groq for AI apps?', 'role': 'assistant', 'name': 'user_proxy'}, {'content': '## Why\u202fGroq\u202fis becoming a go‑to platform for AI applications\n\n| Benefit | What it means for you | Why it matters |\n|---------|----------------------|----------------|\n| **Extreme inference speed** | •\u202fSingle‑cycle tensor streaming →\u202flatency measured in micro‑seconds <br>•\u202fDeterministic, “no‑surprises” execution time | Real‑time use‑cases (search, recommendation, autonomous control, voice assistants) can meet sub‑ms SLA targets that GPUs or TPUs struggle to guarantee. |\n| **Predictable, linear scaling** | •\u202fPerformance grows linearly with the number of Groq cards <br>•\u202fNo “GPU‑style” throttling or memory‑bandwidth contention | Capacity planning is simple—add another card and you get a proportional boost, which is rare on conventional accelerators. |\n|

In [18]:
# ---------------------------------------------------------------------------
# Alternative: Build config list in code (instead of CONFIG_LIST.json)
# ---------------------------------------------------------------------------
# Use this when you prefer to pass the API key from the environment (e.g. .env).
# Validate GROQ_API_KEY so we fail fast with a clear message instead of at API call time.
api_key = os.environ.get("GROQ_API_KEY")
if not api_key or not api_key.strip():
    raise ValueError(
        "GROQ_API_KEY is not set or is empty. "
        "Set it in your environment or in a .env file (e.g. GROQ_API_KEY=gsk_...). "
        "Then re-run the cell that runs load_dotenv() and this cell."
    )

config_list = [{
    "model": "llama-3.3-70b-versatile",
    "api_key": api_key,
    "api_type": "groq"
}]

# Create assistant and user proxy (same as the main example above).
assistant = AssistantAgent(
    name="groq_assistant",
    system_message="You are a helpful AI assistant.",
    llm_config={"config_list": config_list}
)

# human_input_mode="NEVER": no prompts for input; proxy auto-replies so the chat runs unattended.
# max_consecutive_auto_reply: cap auto-replies so the conversation stops after the assistant replies.
user_proxy = UserProxyAgent(
    name="user_proxy",
    code_execution_config=False,
    human_input_mode="NEVER",
    max_consecutive_auto_reply=1,
)

# Initiate the same demo chat.
user_proxy.initiate_chat(
    assistant,
    message="What are the key benefits of using Groq for AI apps?")

user_proxy (to groq_assistant):

What are the key benefits of using Groq for AI apps?

--------------------------------------------------------------------------------
groq_assistant (to user_proxy):

Groq is a chip company that specializes in developing tensor processing units (TPUs) for artificial intelligence (AI) applications. The key benefits of using Groq for AI apps include:

1. **High Performance**: Groq's TPUs are designed to provide high performance and efficiency for AI workloads, allowing for faster processing of complex neural networks and large datasets.
2. **Low Latency**: Groq's architecture is optimized for low latency, which is critical for real-time AI applications such as autonomous vehicles, robotics, and smart home devices.
3. **High Throughput**: Groq's TPUs can handle high-throughput workloads, making them suitable for applications that require processing large amounts of data in parallel, such as natural language processing and computer vision.
4. **Power Effic

/Users/sourav.banerjee/Documents/Codebases/2. AI ENGINEERING/Autogen_Demystified/.venv/lib/python3.13/site-packages/autogen/oai/groq.py:317: UserWarning: Cost calculation not available for model llama-3.3-70b-versatile
  warnings.warn(f"Cost calculation not available for model {model}", UserWarning)


groq_assistant (to user_proxy):

It seems like you didn't ask a question or provide any information for me to respond to. If you have any questions or topics related to Groq or AI, I'd be happy to help. Please feel free to ask, and I'll do my best to provide a helpful response.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (f952e536-eecf-4828-a103-b6585b08a81d): Maximum number of consecutive auto-replies reached


ChatResult(chat_id=35672742558963944655421213888065352268, chat_history=[{'content': 'What are the key benefits of using Groq for AI apps?', 'role': 'assistant', 'name': 'user_proxy'}, {'content': "Groq is a chip company that specializes in developing tensor processing units (TPUs) for artificial intelligence (AI) applications. The key benefits of using Groq for AI apps include:\n\n1. **High Performance**: Groq's TPUs are designed to provide high performance and efficiency for AI workloads, allowing for faster processing of complex neural networks and large datasets.\n2. **Low Latency**: Groq's architecture is optimized for low latency, which is critical for real-time AI applications such as autonomous vehicles, robotics, and smart home devices.\n3. **High Throughput**: Groq's TPUs can handle high-throughput workloads, making them suitable for applications that require processing large amounts of data in parallel, such as natural language processing and computer vision.\n4. **Power Eff

---
## Code execution: assistant + user proxy with an executor

In the next section we switch to a **code-execution** setup: the assistant can suggest code, and the user proxy will run it in a local folder and return the results. This enables tasks like plotting charts or writing files.

The sample `CONFIG_LIST.json` format is shown in the **LLM configuration** section above. Ensure your JSON file exists in the notebook directory (or that the corresponding environment variable is set) before loading the config.

### Build the assistant and user proxy (with code execution)

Here we create:

1. **AssistantAgent** — Same as before, but it will now suggest **code** to accomplish tasks (e.g., plotting, file I/O).
2. **UserProxyAgent** — Configured with a **code executor** so it can run the assistant’s code in a local directory and send the output (or errors) back. The conversation continues until the task is done or you stop it.

In [19]:
# ---------------------------------------------------------------------------
# Create assistant and user proxy with code execution enabled
# ---------------------------------------------------------------------------
# Assistant: same as before; it will now propose code to solve tasks.
assistant = AssistantAgent("assistant", llm_config={"config_list": config_list})

# Executor: runs code in the "coding" subdirectory. All generated scripts
# (e.g. .py files) and outputs (e.g. .png) will appear there.
work_dir = Path("coding")
code_executor = LocalCommandLineCodeExecutor(work_dir=work_dir)

# User proxy: when the assistant sends code, the proxy runs it via code_executor
# and returns stdout/stderr/exit code to the assistant for the next turn.
user_proxy = UserProxyAgent("user_proxy", code_execution_config={"executor": code_executor})

In [20]:
# ---------------------------------------------------------------------------
# Task 1: Stock chart — assistant writes code, user proxy runs it
# ---------------------------------------------------------------------------
# Wrapper to send a natural-language task to the assistant. The user proxy
# starts the chat; the assistant may respond with code; the proxy executes it
# and sends results back until the task is done (or you type 'exit').
def execute_agent(prompt: str):
    """Send a task to the assistant via the user proxy."""
    return user_proxy.initiate_chat(assistant, message=prompt)

# Example 1: Data + plotting. The assistant will typically suggest a Python script
# (e.g. using yfinance + matplotlib), the proxy runs it in the "coding" folder,
# and the plot is saved as stock_price_change.png there.
execute_agent(
    """Plot a chart of Tesla stock price change and compare it to Nvidia's
    stock price change in the same period and save it as stock_price_change.png"""
)

user_proxy (to assistant):

Plot a chart of Tesla stock price change and compare it to Nvidia's
    stock price change in the same period and save it as stock_price_change.png

--------------------------------------------------------------------------------
assistant (to user_proxy):

To plot a chart of Tesla stock price change and compare it to Nvidia's stock price change in the same period, we will use Python with libraries like `yfinance` for stock data and `matplotlib` for plotting. 

First, let's get the stock data for Tesla and Nvidia. We will use the following code:

```python
# filename: stock_plot.py
import yfinance as yf
import matplotlib.pyplot as plt

# Get stock data for Tesla and Nvidia
tesla_data = yf.download('TSLA', period='1y')
nvidia_data = yf.download('NVDA', period='1y')

# Calculate daily stock price change
tesla_data['Price Change'] = tesla_data['Close'].pct_change()
nvidia_data['Price Change'] = nvidia_data['Close'].pct_change()

# Plot the stock price change
pl

KeyboardInterrupt: Interrupted by user

In [ ]:
# ---------------------------------------------------------------------------
# Display the chart produced by the agent (saved in the "coding" work directory)
# ---------------------------------------------------------------------------
Image("coding/stock_price_change.png")

### Task 2: Research report generation

The same assistant + user proxy can handle more involved tasks. In this example, the agent is asked to fetch papers (e.g. from arXiv), summarize them, and write a single Markdown report. The conversation may take several turns (install deps, run scripts, retry on errors). When running in a notebook you may need to press Enter to allow auto-reply or type `exit` to stop.

In [21]:
# ---------------------------------------------------------------------------
# Task 2: Research report — fetch papers, summarize, write one Markdown file
# ---------------------------------------------------------------------------
# The assistant may use arXiv API, write Python scripts to fetch/summarize,
# and iterate if dependencies are missing or execution fails. Can take several minutes.
execute_agent(
    """Fetch 2 papers about using small language models and
    summarize them into a one single research report file named research-report-llms-productivity.md"""
)

user_proxy (to assistant):

Fetch 2 papers about using small language models and
    summarize them into a one single research report file named research-report-llms-productivity.md

--------------------------------------------------------------------------------
assistant (to user_proxy):

To fetch two papers about using small language models and summarize them into a single research report file, I need to perform the following steps:
1. Search for relevant papers using a search engine or an academic database.
2. Download the papers or extract the content.
3. Summarize the papers and create a research report.

First, I will search for papers about using small language models.

```python
# filename: search_papers.py
import requests
from bs4 import BeautifulSoup

def search_papers(query):
    url = "https://scholar.google.com/scholar"
    params = {"q": query, "hl": "en"}
    response = requests.get(url, params=params)
    soup = BeautifulSoup(response.text, 'html.parser')
    papers = 

/Users/sourav.banerjee/Documents/Codebases/2. AI ENGINEERING/Autogen_Demystified/.venv/lib/python3.13/site-packages/autogen/oai/groq.py:317: UserWarning: Cost calculation not available for model llama-3.3-70b-versatile
  warnings.warn(f"Cost calculation not available for model {model}", UserWarning)



>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING CODE BLOCK (inferred language is python)...
user_proxy (to assistant):

exitcode: 0 (execution succeeded)
Code output: Research report file 'research-report-llms-productivity.md' created successfully.


--------------------------------------------------------------------------------
assistant (to user_proxy):

The code executed successfully, and the research report file `research-report-llms-productivity.md` has been created.

The report summarizes the potential of small language models for various natural language processing tasks, highlighting their efficiency, effectiveness, and ease of use. It also provides an overview of two papers that explore the use of small language models for text classification and natural language processing.

You can now review the research report file `research-report-llms-productivity.md` to learn more about the applications and benefits of small language models.

TERMIN

/Users/sourav.banerjee/Documents/Codebases/2. AI ENGINEERING/Autogen_Demystified/.venv/lib/python3.13/site-packages/autogen/oai/groq.py:317: UserWarning: Cost calculation not available for model llama-3.3-70b-versatile
  warnings.warn(f"Cost calculation not available for model {model}", UserWarning)



>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...
user_proxy (to assistant):



--------------------------------------------------------------------------------
assistant (to user_proxy):

TERMINATE

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> TERMINATING RUN (2845a80a-5b3d-4715-beed-6a07b212bb6b): Termination message condition on agent 'user_proxy' met


ChatResult(chat_id=166582752422709567901963683847649901933, chat_history=[{'content': 'Fetch 2 papers about using small language models and\n    summarize them into a one single research report file named research-report-llms-productivity.md', 'role': 'assistant', 'name': 'user_proxy'}, {'content': 'To fetch two papers about using small language models and summarize them into a single research report file, I need to perform the following steps:\n1. Search for relevant papers using a search engine or an academic database.\n2. Download the papers or extract the content.\n3. Summarize the papers and create a research report.\n\nFirst, I will search for papers about using small language models.\n\n```python\n# filename: search_papers.py\nimport requests\nfrom bs4 import BeautifulSoup\n\ndef search_papers(query):\n    url = "https://scholar.google.com/scholar"\n    params = {"q": query, "hl": "en"}\n    response = requests.get(url, params=params)\n    soup = BeautifulSoup(response.text, \'h

In [ ]:
# ---------------------------------------------------------------------------
# Task 2: Research report — fetch papers, summarize, write one Markdown file
# ---------------------------------------------------------------------------
# The assistant may use arXiv API, write Python scripts to fetch/summarize,
# and iterate if dependencies are missing or execution fails. Can take several minutes.
execute_agent(
    """Fetch 3 papers about using large language models to augment human productivity and
    summarize them into a one single research report file named research-report-llms-productivity.md"""
)

In [ ]:
# ---------------------------------------------------------------------------
# Display the generated research report (run after Task 2 completes)
# ---------------------------------------------------------------------------
# The report is saved in the "coding" folder; we read it and render as Markdown.
with open("coding/research-report-llms-productivity.md") as f:
    report = f.read()

Markdown(report)

---

## Summary

You have seen the core AutoGen workflow:

| Step | What you did |
|------|----------------|
| **Config** | Loaded an LLM config from `CONFIG_LIST.json` (or built it in code). |
| **Assistant** | Created an `AssistantAgent` with a system message and `llm_config`. |
| **User proxy** | Created a `UserProxyAgent`; with `code_execution_config` it can run code in a folder. |
| **Chat** | Used `user_proxy.initiate_chat(assistant, message=...)` to run Q&A or code-based tasks. |

**Next steps:** Explore multi-agent patterns (e.g. group chat), custom tools, or different executors. See the [AutoGen documentation](https://microsoft.github.io/autogen/) for more.